# CCD Lookup Example

![CCD Lookup Example](https://proto-bio.github.io/proto-assets/images/tool/ccd_lookup/hero.png)

`ccd-lookup` wraps [`pdbeccdutils`](https://github.com/PDBeurope/ccdutils) for rich enrichment of wwPDB Chemical Component Dictionary entries.

- **Inputs:** CCD codes (e.g. `ATP`) or SMILES strings (auto-detected).
- **Defaults:** fully offline.
- **Optional network features:** UniChem cross-references and PDB usage.

For the microsecond-latency hot path used by Fragment validation, see the in-process `proto_tools.entities.ligands.ccd_utils` module instead.

In [ ]:
from proto_tools.tools.database_retrieval.ccd_lookup import (
    run_ccd_lookup,
    CcdLookupInput,
    CcdLookupConfig,
)

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference

In [ ]:
# Display input docs
display_api_reference("ccd_lookup", "input")

In [ ]:
# Display config docs
display_api_reference("ccd_lookup", "config")

In [ ]:
# Display output docs
display_api_reference("ccd_lookup", "output")

## Example 1 — Enrich a single CCD code (offline)

We look up `ATP` (adenosine triphosphate). All defaults; no network calls.

In [ ]:
inputs = CcdLookupInput(identifiers=["ATP"])
result = run_ccd_lookup(inputs)

entry = result.results[0]
frag = entry.fragment

print(f"CCD code:       {frag.ccd_code}")
print(f"Name:           {frag.name}")
print(f"Formula:        {entry.formula}")
print(f"Formula weight: {entry.formula_weight}")
print(f"SMILES:         {frag.smiles}")
print(f"InChIKey:       {entry.inchikey}")
print(f"# atoms:        {frag.mol.GetNumAtoms()}")
print(f"# bonds:        {frag.mol.GetNumBonds()}")
print(f"Released:       {entry.released}")

## Example 2 — Look up by SMILES (offline)

Acetaminophen / paracetamol → CCD code `TYL`.

In [ ]:
inputs = CcdLookupInput(identifiers=["CC(=O)NC1=CC=C(C=C1)O"])
result = run_ccd_lookup(inputs)

entry = result.results[0]
print(f"Resolved CCD code: {entry.fragment.ccd_code}")
print(f"Name:              {entry.fragment.name}")
print(f"Formula:           {entry.formula}")

## Example 3 — Mixed batch with unresolved entries

Mixing CCD codes and SMILES. The output reports how many resolved and lets you inspect unresolved records.

In [ ]:
inputs = CcdLookupInput(identifiers=[
    "ATP",                               # CCD code
    "HEM",                               # CCD code
    "CC(=O)NC1=CC=C(C=C1)O",             # SMILES → TYL
    "CCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC",  # SMILES with no CCD match
])
result = run_ccd_lookup(inputs)

print(f"Resolved:   {result.num_resolved}")
print(f"Unresolved: {result.num_unresolved}")

for ident, entry in zip(inputs.identifiers, result.results):
    if entry.fragment.ccd_code is None:
        print(f"  {ident!r:40s} → no CCD match")
    else:
        print(f"  {ident!r:40s} → {entry.fragment.ccd_code} ({entry.fragment.name})")

## Example 4 — Network features (opt-in)

Add UniChem cross-references and the list of PDB entries that contain this ligand. Both require network access.

In [ ]:
inputs = CcdLookupInput(identifiers=["ATP"])
config = CcdLookupConfig(
    include_cross_references=True,
    include_pdb_usage=True,
)
result = run_ccd_lookup(inputs, config)

entry = result.results[0]

# UniChem cross-references: source database -> [compound IDs].
print(f"Cross-reference sources ({len(entry.cross_references or {})}):")
for source, ids in (entry.cross_references or {}).items():
    print(f"  {source:18s} {ids[:3]}{'  ...' if len(ids) > 3 else ''}")

# PDB structures containing this ligand. The result list is capped to keep
# the response small — the truncation warning lands on entry.warnings.
print(f"\n# PDB structures with ATP: {len(entry.pdb_structures or [])}")
if entry.pdb_structures:
    print(f"  First few: {entry.pdb_structures[:5]}")

if entry.warnings:
    print(f"\nWarnings: {entry.warnings}")

## Example 5 — Visualize the ligand

Each resolved `Fragment` exposes a lazy RDKit `Mol`. Use RDKit's `Draw` for an inline 2D depiction, and `fragment.visualize()` for an interactive 3D view via py3Dmol.

In [ ]:
from rdkit.Chem import Draw

# Re-enrich ATP for visualization (re-uses the cached subprocess + index).
result = run_ccd_lookup(CcdLookupInput(identifiers=["ATP"]))
atp = result.ligands.fragments[0]

# 2D depiction. Draw.MolToImage strips explicit Hs and computes 2D coords
# automatically; the returned PIL image renders inline.
Draw.MolToImage(atp.mol, size=(400, 400))

> NOTE: The 3D viewer below renders locally (JupyterLab, VS Code) but not in GitHub previews.

In [ ]:
# 3D: Fragment.visualize() generates a conformer and renders an interactive
# py3Dmol viewer. Works for `Ligands` collections too via
# `result.ligands.visualize()` (each fragment gets its own chain).
atp.visualize(width=400, height=400, style="stick")

## Export

`CcdLookupOutput` supports JSON export.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)
result.export(name="ccd_lookup_example", export_path=output_dir, file_format="json")
print(f"Wrote {output_dir}/ccd_lookup_example.json")